# Electricity price streaming — the project at a glance

Direction-of-next-move classification on the NSW electricity market, scored
**inside the event flow**: `Kafka → Flink → SageMaker endpoint → output
topic`, with **Feast** providing the offline training set (the live path
reads each tick's own market state).

Same production lifecycle as the NYC taxi project: frame → collect →
preprocess → explore → store → train → tune → evaluate → deploy → monitor.

## The lifecycle

| Stage | Notebook | What it does |
|---|---|---|
| Frame | `docs/01_problem_framing.md` | Why stream scoring |
| Collect | `docs/02_data_collection.md` | The ELEC2 dataset |
| Preprocess | `docs/03_data_preprocessing.md` | Tables + replayable stream |
| Explore | `eda.ipynb` | Price/demand patterns, drift |
| Feature store | `feature_engineering.ipynb` | Feast apply + materialize |
| Train | `training.ipynb` | CatBoost job on moto S3 |
| Tune | `tuning.ipynb` | depth/learning_rate sweep |
| Evaluate | `evaluation.ipynb` | Holdout AUC gate |
| Deploy | `deploy.ipynb` | The streaming pipeline, live |
| Monitor | `monitoring.ipynb` | Consumer lag + teardown |

## Architecture

```
stream_events.jsonl
        │ (kafka_producer.py)
        ▼
  market-ticks topic ──> Flink: feature vector from tick
                              │ feature vector
                              ▼
                    SageMaker endpoint (CatBoost)
                              ▼
                  price-predictions topic
```

The async property is the *decoupling*: producers never wait on the model,
consumers never wait on producers — Kafka owns the buffers, Flink owns the
mapping, the endpoint owns the scoring.